# Feature Engineering Discovery Loop
Build candidate features, screen them for basic quality/correlation constraints, and run a quick flywheel check.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if str(repo / "python") not in sys.path:
    sys.path.insert(0, str(repo / "python"))

import openquant
import polars as pl


## Hypothesis
Short-horizon momentum plus cross-asset context should improve decision quality versus a naive single-signal baseline.


In [ ]:
ds = openquant.research.make_synthetic_futures_dataset(n_bars=220, seed=19)

base = pl.DataFrame({
    "ts": ds.timestamps,
    "close": ds.close,
    "prob": ds.model_probabilities,
    "side": ds.model_sides,
    "asset_1": [row[1] for row in ds.asset_prices],
    "asset_2": [row[2] for row in ds.asset_prices],
})
base.head(5)


## Candidate Features


In [ ]:
feat = base.with_columns([
    (pl.col("close") / pl.col("close").shift(1) - 1.0).alias("ret_1"),
    (pl.col("close") / pl.col("close").shift(5) - 1.0).alias("ret_5"),
    (pl.col("asset_1") / pl.col("close") - 1.0).alias("xasset_spread_1"),
    (pl.col("asset_2") / pl.col("close") - 1.0).alias("xasset_spread_2"),
    (pl.col("prob") * pl.col("side")).alias("model_edge"),
]).fill_null(0.0)

feat.select(["ret_1","ret_5","xasset_spread_1","xasset_spread_2","model_edge"]).head(8)


## Feature Screening


In [ ]:
screen = openquant.feature_diagnostics.feature_screen_report(
    feat.select(["ret_1","ret_5","xasset_spread_1","xasset_spread_2","model_edge"]),
    min_coverage=0.95,
    max_corr=0.92,
)
screen["table"]


## Quick Flywheel Validation


In [ ]:
out = openquant.research.run_flywheel_iteration(ds, config={"step_size":0.08})
out["summary"]


## Analysis
- Keep only `selected_features` for heavier diagnostics.
- Promote only if net and realized Sharpe gates are both true.


In [ ]:
{
    "selected_features": screen["selected_features"],
    "promotion": out["promotion"],
    "costs": out["costs"],
}
